In [4]:
pip -q install google-adk

In [5]:
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

'AQ.Ab8RN6It5mLMY2OBrtKh4SVFm5-FCe53ibqejDgqW4DmxzVw7g'

In [7]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

# Using the recommended model name
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Hello! We are building a Carbon Coach hackathon agent."
)

print(response.text)

That sounds like a fantastic and timely project! Sustainability paired with agentic workflows makes for a compelling hackathon pitch, especially if the agent moves beyond passive reporting to **proactive, actionable coaching**.

To help you hit the ground running, here are a few ideas, technical angles, and resources you can leverage:

---

### 1. High-Impact Hackathon Angles (Demo "Wow" Factors)
Hackathon judges love things that are interactive and visually clear:

* **Multimodal Receipt/Plate Scanner:** The user uploads a photo of a grocery receipt or a meal, the agent calculates the estimated carbon footprint, and immediately suggests: *"Swapping the beef for chicken or lentils would save 4.2 kg CO₂e. Want a recipe for that?"*
* **Calendar & Travel Optimizer:** Connect to a calendar (or mock one). The agent spots upcoming business trips or commutes, checks real-time public transit vs. driving/flying emissions, and drafts a greener itinerary.
* **Smart Grid / Home Assistant:** Integr

In [8]:
!pip -q install google-adk

In [9]:
import google.adk

print("✅ Google ADK installed successfully!")

✅ Google ADK installed successfully!


In [10]:
# 🌱 Carbon Coach — Emission Factor Database

EMISSION_FACTORS = {
    "travel": {
        "bus": 0.105,          # kg CO2e per km
        "car": 0.192,          # kg CO2e per km
        "motorcycle": 0.103,   # kg CO2e per km
        "train": 0.041,        # kg CO2e per km
        "bicycle": 0.0,
        "walking": 0.0
    },

    "electricity": {
        "grid": 0.82           # kg CO2e per kWh
    },

    "food": {
        "vegetarian": 0.7,     # kg CO2e per meal
        "chicken": 2.5,
        "mutton": 6.0,
        "beef": 7.0,
        "vegan": 0.4
    }
}

EMISSION_FACTOR_SOURCE = (
    "Prototype emission factors used for the DevFest hackathon. "
    "Factors are explicitly stored in the application and are "
    "used by deterministic Python calculation tools."
)

print("✅ Carbon emission database loaded!")
print(EMISSION_FACTORS)

✅ Carbon emission database loaded!
{'travel': {'bus': 0.105, 'car': 0.192, 'motorcycle': 0.103, 'train': 0.041, 'bicycle': 0.0, 'walking': 0.0}, 'electricity': {'grid': 0.82}, 'food': {'vegetarian': 0.7, 'chicken': 2.5, 'mutton': 6.0, 'beef': 7.0, 'vegan': 0.4}}


In [11]:
def calculate_travel_emissions(mode: str, km: float) -> dict:
    """
    Calculate carbon emissions from travel.

    mode: bus, car, motorcycle, train, bicycle, walking
    km: distance travelled in kilometres
    """

    mode = mode.lower().strip()

    # Check whether travel mode is supported
    if mode not in EMISSION_FACTORS["travel"]:
        return {
            "success": False,
            "error": f"Unsupported travel mode: {mode}",
            "supported_modes": list(EMISSION_FACTORS["travel"].keys())
        }

    # Prevent invalid distance
    if km < 0:
        return {
            "success": False,
            "error": "Distance cannot be negative."
        }

    # Get emission factor
    factor = EMISSION_FACTORS["travel"][mode]

    # Deterministic calculation
    emissions = km * factor

    return {
        "success": True,
        "category": "travel",
        "mode": mode,
        "distance_km": km,
        "emission_factor_kg_per_km": factor,
        "emissions_kg_co2e": round(emissions, 3),
        "source": EMISSION_FACTOR_SOURCE
    }

In [12]:
result = calculate_travel_emissions("bus", 12)

print(result)

{'success': True, 'category': 'travel', 'mode': 'bus', 'distance_km': 12, 'emission_factor_kg_per_km': 0.105, 'emissions_kg_co2e': 1.26, 'source': 'Prototype emission factors used for the DevFest hackathon. Factors are explicitly stored in the application and are used by deterministic Python calculation tools.'}


In [13]:
def calculate_electricity_emissions(kwh: float) -> dict:
    """
    Calculate carbon emissions from electricity consumption.

    kwh: electricity consumed in kilowatt-hours
    """

    # Validate input
    if kwh < 0:
        return {
            "success": False,
            "error": "Electricity consumption cannot be negative."
        }

    # Get electricity emission factor
    factor = EMISSION_FACTORS["electricity"]["grid"]

    # Deterministic calculation
    emissions = kwh * factor

    return {
        "success": True,
        "category": "electricity",
        "energy_kwh": kwh,
        "emission_factor_kg_per_kwh": factor,
        "emissions_kg_co2e": round(emissions, 3),
        "source": EMISSION_FACTOR_SOURCE
    }

In [14]:
import os
from google.colab import userdata

# Export the API key to environment variables so google-adk's internal client can find it
os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# Run the deterministic test to verify
result = calculate_electricity_emissions(4)
print("Deterministic calculation check:", result)

Deterministic calculation check: {'success': True, 'category': 'electricity', 'energy_kwh': 4, 'emission_factor_kg_per_kwh': 0.82, 'emissions_kg_co2e': 3.28, 'source': 'Prototype emission factors used for the DevFest hackathon. Factors are explicitly stored in the application and are used by deterministic Python calculation tools.'}


In [15]:
def calculate_food_emissions(meal_type: str) -> dict:
    """
    Calculate estimated carbon emissions for a meal.

    Supported meal types:
    vegetarian, chicken, mutton, beef, vegan
    """

    meal_type = meal_type.lower().strip()

    # Check supported meal type
    if meal_type not in EMISSION_FACTORS["food"]:
        return {
            "success": False,
            "error": f"Unsupported meal type: {meal_type}",
            "supported_meals": list(EMISSION_FACTORS["food"].keys())
        }

    # Get emission factor
    factor = EMISSION_FACTORS["food"][meal_type]

    # One meal = one emission-factor unit
    emissions = factor

    return {
        "success": True,
        "category": "food",
        "meal_type": meal_type,
        "emission_factor_kg_per_meal": factor,
        "emissions_kg_co2e": round(emissions, 3),
        "source": EMISSION_FACTOR_SOURCE
    }

In [16]:
result = calculate_food_emissions("chicken")

print(result)

{'success': True, 'category': 'food', 'meal_type': 'chicken', 'emission_factor_kg_per_meal': 2.5, 'emissions_kg_co2e': 2.5, 'source': 'Prototype emission factors used for the DevFest hackathon. Factors are explicitly stored in the application and are used by deterministic Python calculation tools.'}


In [17]:
print(calculate_food_emissions("vegetarian"))


{'success': True, 'category': 'food', 'meal_type': 'vegetarian', 'emission_factor_kg_per_meal': 0.7, 'emissions_kg_co2e': 0.7, 'source': 'Prototype emission factors used for the DevFest hackathon. Factors are explicitly stored in the application and are used by deterministic Python calculation tools.'}


In [18]:
from datetime import datetime

# Stores activities during the current Carbon Coach session
activity_log = []

print("🧠 Carbon Coach memory initialized!")

🧠 Carbon Coach memory initialized!


In [19]:
def log_activity(category: str, details: str, emissions_kg: float) -> dict:
    """
    Save a calculated activity into Carbon Coach memory.
    """

    activity = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "category": category,
        "details": details,
        "emissions_kg_co2e": round(emissions_kg, 3)
    }

    activity_log.append(activity)

    return {
        "success": True,
        "message": "Activity saved successfully.",
        "activity": activity
    }

In [20]:
result = log_activity(
    category="travel",
    details="Travelled 12 km by bus",
    emissions_kg=1.26
)

print(result)

{'success': True, 'message': 'Activity saved successfully.', 'activity': {'timestamp': '2026-10-30T11:18:28', 'category': 'travel', 'details': 'Travelled 12 km by bus', 'emissions_kg_co2e': 1.26}}


In [21]:
print("Current activities:")
print(activity_log)

Current activities:
[{'timestamp': '2026-10-30T11:18:28', 'category': 'travel', 'details': 'Travelled 12 km by bus', 'emissions_kg_co2e': 1.26}]


In [22]:
def get_summary() -> dict:
    """
    Summarize all activities stored in the current session.
    """

    # No activities yet
    if not activity_log:
        return {
            "success": True,
            "total_kg_co2e": 0,
            "activity_count": 0,
            "category_totals": {},
            "biggest_source": None,
            "activities": [],
            "message": "No activities have been logged yet."
        }

    # Calculate total emissions
    total = sum(
        activity["emissions_kg_co2e"]
        for activity in activity_log
    )

    # Calculate emissions by category
    category_totals = {}

    for activity in activity_log:
        category = activity["category"]

        category_totals[category] = (
            category_totals.get(category, 0)
            + activity["emissions_kg_co2e"]
        )

    # Find biggest source
    biggest_source = max(
        category_totals,
        key=category_totals.get
    )

    return {
        "success": True,
        "total_kg_co2e": round(total, 3),
        "activity_count": len(activity_log),
        "category_totals": {
            category: round(value, 3)
            for category, value in category_totals.items()
        },
        "biggest_source": biggest_source,
        "activities": activity_log
    }

In [23]:
summary = get_summary()

print(summary)

{'success': True, 'total_kg_co2e': 1.26, 'activity_count': 1, 'category_totals': {'travel': 1.26}, 'biggest_source': 'travel', 'activities': [{'timestamp': '2026-10-30T11:18:28', 'category': 'travel', 'details': 'Travelled 12 km by bus', 'emissions_kg_co2e': 1.26}]}


In [24]:
# Add electricity activity
electricity_result = calculate_electricity_emissions(4)

log_activity(
    category="electricity",
    details="Used 4 kWh electricity",
    emissions_kg=electricity_result["emissions_kg_co2e"]
)

# Add food activity
food_result = calculate_food_emissions("chicken")

log_activity(
    category="food",
    details="Ate a chicken meal",
    emissions_kg=food_result["emissions_kg_co2e"]
)

print("✅ Activities added!")

✅ Activities added!


In [25]:
summary = get_summary()

print("🌱 CARBON COACH SUMMARY")
print("-----------------------")

print(
    "Total emissions:",
    summary["total_kg_co2e"],
    "kg CO₂e"
)

print(
    "Activities:",
    summary["activity_count"]
)

print(
    "Category totals:",
    summary["category_totals"]
)

print(
    "Biggest source:",
    summary["biggest_source"]
)

🌱 CARBON COACH SUMMARY
-----------------------
Total emissions: 7.04 kg CO₂e
Activities: 3
Category totals: {'travel': 1.26, 'electricity': 3.28, 'food': 2.5}
Biggest source: electricity


In [26]:
from google.adk.agents import Agent

root_agent = Agent(
    name="carbon_coach",

    model="gemini-3.8-flash",

    description="""
    Carbon Coach is an AI agent that helps users measure,
    understand, and reduce their carbon footprint.
    """,

    instruction="""
You are Carbon Coach, an AI assistant that helps users
measure, understand, and reduce their carbon footprint.

YOUR CORE RULES:

1. NEVER invent carbon emission numbers.

2. Whenever a user asks for a carbon calculation,
   ALWAYS use the appropriate Python calculation tool.

3. Do NOT calculate emissions yourself.

4. All numerical carbon results must come from tool output.

5. When a user reports TRAVEL:
   - identify the travel mode and distance
   - call calculate_travel_emissions
   - save the activity using log_activity
   - tell the user the calculated emissions

6. When a user reports ELECTRICITY:
   - identify electricity consumption in kWh
   - call calculate_electricity_emissions
   - save the activity using log_activity
   - tell the user the calculated emissions

7. When a user reports FOOD:
   - identify the meal type
   - call calculate_food_emissions
   - save the activity using log_activity
   - tell the user the calculated emissions

8. When the user asks for their total emissions,
   call get_summary.

9. When the user asks about their biggest source,
   call get_summary.

10. When information required for a calculation is missing,
    ASK the user for the missing information.
    Never guess.

11. Clearly explain that emission factors are estimates
    when appropriate.

12. Be concise, friendly, and practical.

13. Give actionable suggestions based on the user's
    actual logged activities.

14. Never claim that you measured something that you
    did not actually measure.

EXAMPLE:

User:
"I travelled 12 km by bus today."

You should:
1. Call calculate_travel_emissions.
2. Call log_activity with the result.
3. Tell the user the calculated kg CO2e.

User:
"How much did I produce today?"

You should:
1. Call get_summary.
2. Report the total from the tool.
3. Mention the biggest source if available.

Remember:
YOU ARE THE COACH.
THE PYTHON TOOLS ARE THE CALCULATION ENGINE.
NEVER MAKE UP NUMBERS.
""",

    tools=[
        calculate_travel_emissions,
        calculate_electricity_emissions,
        calculate_food_emissions,
        log_activity,
        get_summary
    ]
)

print("✅ Carbon Coach ADK agent created with gemini-3.8-flash!")

✅ Carbon Coach ADK agent created with gemini-3.8-flash!


In [27]:
tools=[
    calculate_travel_emissions,
    calculate_electricity_emissions,
    calculate_food_emissions,
    log_activity,
    get_summary
]

In [28]:
from google.adk.runners import InMemoryRunner

# Recreate the runner using the updated agent and environment keys
runner = InMemoryRunner(
    agent=root_agent,
    app_name="carbon_coach"
)

print("✅ Carbon Coach runner successfully updated!")

✅ Carbon Coach runner successfully updated!


In [29]:
session = await runner.session_service.create_session(
    app_name="carbon_coach",
    user_id="demo_user"
)

print("✅ Fresh session created successfully!")
print("Session ID:", session.id)

✅ Fresh session created successfully!
Session ID: d095e9a0-df01-4d99-bca1-90852da4fb2c


In [30]:
from google.genai import types

message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="I travelled 12 km by bus today."
        )
    ]
)

print("👤 USER:")
print("I travelled 12 km by bus today.")
print("\n🤖 CARBON COACH:")

👤 USER:
I travelled 12 km by bus today.

🤖 CARBON COACH:


In [31]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(
    agent=root_agent,
    app_name="carbon_coach"
)

print("✅ Runner created successfully!")

✅ Runner created successfully!


In [32]:
session = await runner.session_service.create_session(
    app_name="carbon_coach",
    user_id="demo_user"
)

print("✅ Session created!")
print("Session ID:", session.id)

✅ Session created!
Session ID: 74421403-88e6-4676-bafa-d395ad8a4c51


In [33]:
from google.genai import types

message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="I travelled 12 km by bus today."
        )
    ]
)

print("✅ Message created!")


✅ Message created!


In [35]:
from google.adk.agents import Agent

root_agent = Agent(
    name="carbon_coach",
    model="gemini-2.5-flash-lite",
    description="""
    Carbon Coach is an AI agent that helps users measure,
    understand, and reduce their carbon footprint.
    """,
    instruction="""
You are Carbon Coach, an AI assistant that helps users
measure, understand, and reduce their carbon footprint.

YOUR CORE RULES:

1. NEVER invent carbon emission numbers.

2. Whenever a user asks for a carbon calculation,
   ALWAYS use the appropriate Python calculation tool.

3. Do NOT calculate emissions yourself.

4. All numerical carbon results must come from tool output.

5. When a user reports TRAVEL:
   - identify the travel mode and distance
   - call calculate_travel_emissions
   - save the activity using log_activity
   - tell the user the calculated emissions

6. When a user reports ELECTRICITY:
   - identify electricity consumption in kWh
   - call calculate_electricity_emissions
   - save the activity using log_activity
   - tell the user the calculated emissions

7. When a user reports FOOD:
   - identify the meal type
   - call calculate_food_emissions
   - save the activity using log_activity
   - tell the user the calculated emissions

8. When the user asks for their total emissions,
   call get_summary.

9. When the user asks about their biggest source,
   call get_summary.

10. When information required for a calculation is missing,
    ASK the user for the missing information.
    Never guess.

11. Clearly explain that emission factors are estimates
    when appropriate.

12. Be concise, friendly, and practical.

13. Give actionable suggestions based on the user's
    actual logged activities.

14. Never claim that you measured something that you
    did not actually measure.

Remember:
YOU ARE THE COACH.
THE PYTHON TOOLS ARE THE CALCULATION ENGINE.
NEVER MAKE UP NUMBERS.
""",
    tools=[
        calculate_travel_emissions,
        calculate_electricity_emissions,
        calculate_food_emissions,
        log_activity,
        get_summary
    ]
)

print("✅ Carbon Coach ADK agent created!")

✅ Carbon Coach ADK agent created!


In [37]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(
    agent=root_agent,
    app_name="carbon_coach"
)

print("✅ Runner created successfully!")

✅ Runner created successfully!


In [39]:
session = await runner.session_service.create_session(
    app_name="carbon_coach",
    user_id="demo_user"
)

print("✅ Session created!")
print("Session ID:", session.id)

✅ Session created!
Session ID: 7fecf854-1938-4ad1-ae54-c9cea1a1520e


In [41]:
from google.genai import types

message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="I travelled 12 km by bus today."
        )
    ]
)

print("✅ Message created!")

✅ Message created!


In [45]:
from google.adk.agents import Agent

root_agent = Agent(
    name="carbon_coach",
    model="gemini-3.5-flash-lite",
    description="""
    Carbon Coach is an AI agent that helps users measure,
    understand, and reduce their carbon footprint.
    """,
    instruction="""
You are Carbon Coach, an AI assistant that helps users
measure, understand, and reduce their carbon footprint.

YOUR CORE RULES:

1. NEVER invent carbon emission numbers.
2. Whenever a user asks for a carbon calculation,
   ALWAYS use the appropriate Python calculation tool.
3. Do NOT calculate emissions yourself.
4. All numerical carbon results must come from tool output.

5. For TRAVEL:
   - identify mode and distance
   - call calculate_travel_emissions
   - save using log_activity
   - report the tool result

6. For ELECTRICITY:
   - identify kWh
   - call calculate_electricity_emissions
   - save using log_activity
   - report the tool result

7. For FOOD:
   - identify meal type
   - call calculate_food_emissions
   - save using log_activity
   - report the tool result

8. For total emissions, call get_summary.
9. For biggest source, call get_summary.
10. If required information is missing, ask the user.
11. Never guess missing values.
12. Clearly explain that emission factors are estimates.
13. Give practical suggestions based on logged activities.

YOU ARE THE COACH.
THE PYTHON TOOLS ARE THE CALCULATION ENGINE.
NEVER MAKE UP NUMBERS.
""",
    tools=[
        calculate_travel_emissions,
        calculate_electricity_emissions,
        calculate_food_emissions,
        log_activity,
        get_summary
    ]
)

print("✅ Carbon Coach agent recreated with Gemini 3.5 Flash Lite!")

✅ Carbon Coach agent recreated with Gemini 3.5 Flash Lite!


In [46]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(
    agent=root_agent,
    app_name="carbon_coach"
)

print("✅ Runner updated with Gemini 3.5 Flash Lite!")

✅ Runner updated with Gemini 3.5 Flash Lite!


In [48]:
async for event in runner.run_async(
    user_id="demo_user",
    session_id=session.id,
    new_message=message
):
    if event.is_final_response():
        print("🤖 CARBON COACH:")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)
        else:
            print("⚠️ No response received.")

SessionNotFoundError: Session not found: 7fecf854-1938-4ad1-ae54-c9cea1a1520e

In [50]:
session = await runner.session_service.create_session(
    app_name="carbon_coach",
    user_id="demo_user"
)

print("✅ New session created!")
print("Session ID:", session.id)

✅ New session created!
Session ID: e313cb57-b7b8-4b91-af6a-67b847f0a29e


In [52]:
async for event in runner.run_async(
    user_id="demo_user",
    session_id=session.id,
    new_message=message
):
    if event.is_final_response():
        print("🤖 CARBON COACH:")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)
        else:
            print("⚠️ No response received.")

🤖 CARBON COACH:
You travelled 12 km by bus today, resulting in an estimated carbon emission of **1.26 kg CO₂e**. 

*(Note: Emission factors are estimates.)*

**Practical Suggestion:** 
Taking public transport like the bus is already a great choice! If you ever want to reduce this further for shorter journeys, walking or cycling are great zero-emission alternatives.

I've logged this activity for your session. Let me know if you have any other activities you'd like to record or if you'd like to see your summary!


In [54]:
from google.genai import types

message2 = types.Content(
    role="user",
    parts=[
        types.Part(
            text="How much carbon have I produced so far?"
        )
    ]
)

async for event in runner.run_async(
    user_id="demo_user",
    session_id=session.id,
    new_message=message2
):
    if event.is_final_response():
        print("🤖 CARBON COACH:")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)

🤖 CARBON COACH:
You have produced a total of **9.56 kg CO₂e** so far across your logged activities.

*(Note: Emission factors are estimates.)*

**Category Breakdown:**
* **Travel:** 3.78 kg CO₂e
* **Electricity:** 3.28 kg CO₂e
* **Food:** 2.50 kg CO₂e

**Practical Suggestion:** 
Your biggest source of emissions is currently **travel**. To help lower this, you could continue utilizing public transit like the bus, or look into carpooling, cycling, or walking for upcoming trips!


In [55]:
def suggest_reductions() -> dict:
    if not activity_log:
        return {
            "success": True,
            "suggestions": [],
            "message": "No activities logged yet."
        }

    suggestions = []

    for activity in activity_log:
        category = activity["category"]
        details = activity["details"].lower()
        emissions = activity["emissions_kg_co2e"]

        # Travel: car → bus
        if category == "travel" and "car" in details:
            saving = emissions - (
                emissions
                * EMISSION_FACTORS["travel"]["bus"]
                / EMISSION_FACTORS["travel"]["car"]
            )

            suggestions.append({
                "action": "Use a bus instead of a car for the same distance",
                "estimated_saving_kg_co2e": round(max(0, saving), 3)
            })

        # Electricity: reduce usage by 20%
        elif category == "electricity":
            saving = emissions * 0.20

            suggestions.append({
                "action": "Reduce unnecessary electricity use by approximately 20%",
                "estimated_saving_kg_co2e": round(saving, 3)
            })

        # Food: chicken → vegetarian
        elif category == "food" and "chicken" in details:
            saving = (
                EMISSION_FACTORS["food"]["chicken"]
                - EMISSION_FACTORS["food"]["vegetarian"]
            )

            suggestions.append({
                "action": "Replace one chicken meal with a vegetarian meal",
                "estimated_saving_kg_co2e": round(max(0, saving), 3)
            })

    return {
        "success": True,
        "suggestions": suggestions,
        "message": "Suggestions are based on the user's logged activities."
    }

print("✅ Reduction suggestion tool created!")

✅ Reduction suggestion tool created!


In [56]:
root_agent.tools.append(suggest_reductions)

print("✅ Reduction suggestion tool added to Carbon Coach!")

✅ Reduction suggestion tool added to Carbon Coach!


In [58]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(
    agent=root_agent,
    app_name="carbon_coach"
)

print("✅ Runner updated with reduction tool!")

✅ Runner updated with reduction tool!


In [60]:
session = await runner.session_service.create_session(
    app_name="carbon_coach",
    user_id="demo_user"
)

print("✅ New session created!")
print("Session ID:", session.id)

✅ New session created!
Session ID: cd9cf926-86d2-4a5e-9286-28ca797bd4ca


In [61]:
message3 = types.Content(
    role="user",
    parts=[
        types.Part(
            text="Based on my logged activities, how can I reduce my carbon footprint? Give me specific actions and estimated savings."
        )
    ]
)

async for event in runner.run_async(
    user_id="demo_user",
    session_id=session.id,
    new_message=message3
):
    if event.is_final_response():
        print("🤖 CARBON COACH:")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)

🤖 CARBON COACH:
Based on your logged activities so far, here is how you can reduce your carbon footprint. 

### Your Logged Summary
* **Total Emissions:** 9.56 kg CO₂e
* **Biggest Source:** Travel (3.78 kg CO₂e across your bus trips)

*(Note: All emission factors are estimates.)*

---

### Specific Reduction Actions & Estimated Savings

1. **Dietary Adjustments:**
   * **Action:** Replace one chicken meal with a vegetarian option.
   * **Estimated Saving:** **1.8 kg CO₂e**

2. **Energy Efficiency:**
   * **Action:** Reduce your electricity consumption by approximately 20% (e.g., turning off unused lights, unplugging appliances on standby).
   * **Estimated Saving:** **0.656 kg CO₂e**

3. **Travel & Commuting:**
   * **Action:** Since travel is your biggest source of emissions (even by bus), consider walking or cycling for some of your shorter 12 km trips if feasible.


In [63]:
def suggest_reductions() -> dict:
    if not activity_log:
        return {
            "success": True,
            "suggestions": [],
            "message": "No activities logged yet."
        }

    suggestions = []

    for activity in activity_log:
        category = activity["category"]
        details = activity["details"].lower()
        emissions = activity["emissions_kg_co2e"]

        # TRAVEL
        if category == "travel":

            if "car" in details:
                saving = emissions - (
                    emissions
                    * EMISSION_FACTORS["travel"]["bus"]
                    / EMISSION_FACTORS["travel"]["car"]
                )

                suggestions.append({
                    "action": "Use a bus instead of a car for the same distance.",
                    "estimated_saving_kg_co2e": round(max(0, saving), 3)
                })

            elif "bus" in details:
                # Compare bus with bicycle for the same distance
                distance = activity["emissions_kg_co2e"] / EMISSION_FACTORS["travel"]["bus"]

                saving = (
                    calculate_travel_emissions("bus", distance)["emissions_kg_co2e"]
                    - calculate_travel_emissions("bicycle", distance)["emissions_kg_co2e"]
                )

                suggestions.append({
                    "action": f"For suitable shorter trips, consider cycling instead of the bus ({round(distance, 1)} km example).",
                    "estimated_saving_kg_co2e": round(max(0, saving), 3)
                })

        # ELECTRICITY
        elif category == "electricity":

            saving = emissions * 0.20

            suggestions.append({
                "action": "Reduce unnecessary electricity consumption by approximately 20%.",
                "estimated_saving_kg_co2e": round(saving, 3)
            })

        # FOOD
        elif category == "food" and "chicken" in details:

            saving = (
                EMISSION_FACTORS["food"]["chicken"]
                - EMISSION_FACTORS["food"]["vegetarian"]
            )

            suggestions.append({
                "action": "Replace one chicken meal with a vegetarian meal.",
                "estimated_saving_kg_co2e": round(max(0, saving), 3)
            })

    return {
        "success": True,
        "suggestions": suggestions,
        "message": "Suggestions are based on logged activities and the application's emission factors."
    }

print("✅ Reduction tool updated!")

✅ Reduction tool updated!


In [64]:
# Replace the old reduction tool with the updated one
root_agent.tools = [
    calculate_travel_emissions,
    calculate_electricity_emissions,
    calculate_food_emissions,
    log_activity,
    get_summary,
    suggest_reductions
]

print("✅ Updated reduction tool connected to Carbon Coach!")

✅ Updated reduction tool connected to Carbon Coach!


In [65]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(
    agent=root_agent,
    app_name="carbon_coach"
)

print("✅ Runner updated with the latest tools!")

✅ Runner updated with the latest tools!


In [66]:
session = await runner.session_service.create_session(
    app_name="carbon_coach",
    user_id="demo_user"
)

print("✅ Final test session created!")

✅ Final test session created!


In [67]:
message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="Based on my logged activities, give me 3 specific ways to reduce my carbon footprint and estimate the savings for each."
        )
    ]
)

async for event in runner.run_async(
    user_id="demo_user",
    session_id=session.id,
    new_message=message
):
    if event.is_final_response():
        print("🤖 CARBON COACH:\n")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)

🤖 CARBON COACH:

Based on your logged activities, here are 3 specific ways you can reduce your carbon footprint:

1. **Active Travel for Shorter Trips**
   * **Suggestion:** Consider cycling instead of taking the bus for suitable shorter trips (such as a 12.0 km trip).
   * **Estimated Saving:** ~1.26 kg CO₂e

2. **Energy Conservation**
   * **Suggestion:** Reduce your overall electricity consumption by approximately 20% by turning off lights, unplugging idle devices, or lowering heating/cooling usage.
   * **Estimated Saving:** ~0.656 kg CO₂e

3. **Dietary Adjustments**
   * **Suggestion:** Replace one of your chicken meals with a vegetarian meal.
   * **Estimated Saving:** ~1.8 kg CO₂e

*Please note that these emission factors and savings are estimates.*


In [68]:

def compare_travel_modes(
    km: float,
    current_mode: str,
    alternative_mode: str
) -> dict:

    current = calculate_travel_emissions(current_mode, km)
    alternative = calculate_travel_emissions(alternative_mode, km)

    if not current["success"]:
        return {
            "success": False,
            "error": current["error"]
        }

    if not alternative["success"]:
        return {
            "success": False,
            "error": alternative["error"]
        }

    saving = (
        current["emissions_kg_co2e"]
        - alternative["emissions_kg_co2e"]
    )

    return {
        "success": True,
        "distance_km": km,
        "current_mode": current_mode,
        "current_emissions_kg_co2e": current["emissions_kg_co2e"],
        "alternative_mode": alternative_mode,
        "alternative_emissions_kg_co2e": alternative["emissions_kg_co2e"],
        "potential_saving_kg_co2e": round(max(0, saving), 3)
    }

print("✅ What-If travel comparison tool added!")

✅ What-If travel comparison tool added!


In [69]:

root_agent.tools = [
    calculate_travel_emissions,
    calculate_electricity_emissions,
    calculate_food_emissions,
    log_activity,
    get_summary,
    suggest_reductions,
    compare_travel_modes
]

print("✅ Carbon Coach updated with the What-If comparison tool!")

✅ Carbon Coach updated with the What-If comparison tool!


In [70]:

from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(
    agent=root_agent,
    app_name="carbon_coach"
)

print("✅ Runner updated with What-If tool!")


✅ Runner updated with What-If tool!


In [72]:

session = await runner.session_service.create_session(
    app_name="carbon_coach",
    user_id="demo_user"
)

print("✅ Fresh session created!")
print("Session ID:", session.id)

✅ Fresh session created!
Session ID: b2fb2a4e-0408-4432-a4e8-a6698d7b6f3c


In [73]:

message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="What if I travelled 12 km by bicycle instead of bus? How much carbon would I save?"
        )
    ]
)

async for event in runner.run_async(
    user_id="demo_user",
    session_id=session.id,
    new_message=message
):
    if event.is_final_response():
        print("🤖 CARBON COACH:\n")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)

🤖 CARBON COACH:

If you travelled 12 km by bicycle instead of a bus, you would save **1.26 kg of CO₂e** (reducing your emissions from 1.26 kg CO₂e by bus down to 0 kg CO₂e by bicycle).

*Please note that these emission factors are estimates.*


In [75]:

def run_carbon_tests():

    tests = [
        {
            "name": "Bus calculation",
            "result": calculate_travel_emissions("bus", 10),
            "expected": 1.05
        },
        {
            "name": "Electricity calculation",
            "result": calculate_electricity_emissions(10),
            "expected": 8.2
        },
        {
            "name": "Zero-emission cycling",
            "result": calculate_travel_emissions("bicycle", 10),
            "expected": 0
        }
    ]

    passed = 0

    for test in tests:
        actual = test["result"]["emissions_kg_co2e"]

        if actual == test["expected"]:
            print("✅", test["name"])
            passed += 1
        else:
            print(
                "❌", test["name"],
                "| Expected:", test["expected"],
                "| Got:", actual
            )

    print("\n" + "=" * 40)
    print(f"RESULT: {passed}/{len(tests)} tests passed")
    print("=" * 40)


run_carbon_tests()

✅ Bus calculation
✅ Electricity calculation
✅ Zero-emission cycling

RESULT: 3/3 tests passed


In [76]:

EMISSION_FACTORS["waste"] = {
    "general": 0.5,
    "recycled": 0.1
}

def calculate_waste_emissions(
    waste_type: str,
    kg: float
) -> dict:

    waste_type = waste_type.lower().strip()

    if waste_type not in EMISSION_FACTORS["waste"]:
        return {
            "success": False,
            "error": f"Unsupported waste type: {waste_type}",
            "supported_types": list(EMISSION_FACTORS["waste"].keys())
        }

    if kg < 0:
        return {
            "success": False,
            "error": "Waste amount cannot be negative."
        }

    factor = EMISSION_FACTORS["waste"][waste_type]
    emissions = kg * factor

    return {
        "success": True,
        "category": "waste",
        "waste_type": waste_type,
        "waste_kg": kg,
        "emission_factor_kg_co2e_per_kg": factor,
        "emissions_kg_co2e": round(emissions, 3),
        "source": EMISSION_FACTOR_SOURCE
    }

print("✅ Waste category added!")

✅ Waste category added!


In [77]:

root_agent.tools = [
    calculate_travel_emissions,
    calculate_electricity_emissions,
    calculate_food_emissions,
    calculate_waste_emissions,
    log_activity,
    get_summary,
    suggest_reductions,
    compare_travel_modes
]

print("✅ Waste calculation tool connected to Carbon Coach!")

✅ Waste calculation tool connected to Carbon Coach!


In [78]:

from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(
    agent=root_agent,
    app_name="carbon_coach"
)

print("✅ Runner updated with Waste tool!")

✅ Runner updated with Waste tool!


In [79]:

session = await runner.session_service.create_session(
    app_name="carbon_coach",
    user_id="demo_user"
)

print("✅ Fresh session created!")
print("Session ID:", session.id)

✅ Fresh session created!
Session ID: 8ffecbd3-f32f-436d-b3a1-9196777ec921


In [80]:

message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="I generated 2 kg of general waste today. Calculate my carbon emissions from it and save the activity."
        )
    ]
)

async for event in runner.run_async(
    user_id="demo_user",
    session_id=session.id,
    new_message=message
):
    if event.is_final_response():
        print("🤖 CARBON COACH:\n")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)

🤖 CARBON COACH:

Based on the calculation using our estimation factors, generating 2 kg of general waste produces **1.0 kg of CO₂e** (carbon dioxide equivalent). 

*Please note that this emission factor is an estimate.*

I have successfully logged this activity to your Carbon Coach session!


In [81]:

message = types.Content(
    role="user",
    parts=[
        types.Part(
            text=(
                "Today I travelled 10 km by bus, "
                "used 5 kWh of electricity, "
                "ate a chicken meal, "
                "and generated 2 kg of general waste. "
                "Calculate and save all my activities, "
                "then tell me my total carbon footprint."
            )
        )
    ]
)

async for event in runner.run_async(
    user_id="demo_user",
    session_id=session.id,
    new_message=message
):
    if event.is_final_response():
        print("🤖 CARBON COACH:\n")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)


🤖 CARBON COACH:

I have calculated and successfully saved all your new activities. Here are the breakdowns for your recent actions (note that emission factors are estimates):

1. **Travel (10 km by bus):** **1.05 kg CO₂e**
2. **Electricity (5 kWh):** **4.1 kg CO₂e**
3. **Food (Chicken meal):** **2.5 kg CO₂e**
4. **Waste (2 kg general waste):** **1.0 kg CO₂e**

### Total Carbon Footprint Summary
Based on all activities recorded in your current session (including previous entries), your total footprint is **19.21 kg CO₂e** across 10 logged activities. 

Your biggest overall emission source is **electricity**.


In [82]:
!pip -q install gradio

import gradio as gr

print("✅ Gradio installed!")

✅ Gradio installed!


In [83]:

async def carbon_coach_chat(user_message, history):

    message = types.Content(
        role="user",
        parts=[
            types.Part(text=user_message)
        ]
    )

    response_text = ""

    async for event in runner.run_async(
        user_id="demo_user",
        session_id=session.id,
        new_message=message
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        response_text += part.text

    return response_text


demo = gr.ChatInterface(
    fn=carbon_coach_chat,
    title="🌱 Carbon Coach",
    description=(
        "AI-powered carbon footprint coach — "
        "track activities, calculate emissions, "
        "and discover ways to reduce your footprint."
    ),
    examples=[
        "I travelled 12 km by bus today.",
        "I used 4 kWh of electricity today.",
        "I ate chicken for lunch.",
        "I generated 2 kg of general waste.",
        "How much carbon have I produced?",
        "How can I reduce my carbon footprint?",
        "What if I travelled 10 km by bicycle instead of bus?"
    ]
)

print("✅ Carbon Coach UI created!")

✅ Carbon Coach UI created!


In [84]:

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d3feacd2c2ad5ec80d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [85]:
demo.launch(
    share=True,
    debug=True,
    show_error=True
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d3feacd2c2ad5ec80d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d3feacd2c2ad5ec80d.gradio.live


In [ ]:

import gradio as gr

test_demo = gr.Interface(
    fn=lambda x: f"Carbon Coach is alive! 🌱 You entered: {x}",
    inputs=gr.Textbox(label="Test"),
    outputs=gr.Textbox(label="Response"),
    title="🌱 Carbon Coach"
)

test_demo.launch(
    share=True,
    debug=True
)

In [88]:

async def carbon_coach_chat(user_message, history):

    message = types.Content(
        role="user",
        parts=[
            types.Part(text=user_message)
        ]
    )

    response_text = ""

    try:
        async for event in runner.run_async(
            user_id="demo_user",
            session_id=session.id,
            new_message=message
        ):
            if event.is_final_response():
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            response_text += part.text

        return response_text if response_text else "⚠️ No response received from Carbon Coach."

    except Exception as e:
        return f"⚠️ Carbon Coach error: {str(e)}"


demo = gr.Interface(
    fn=carbon_coach_chat,
    inputs=gr.Textbox(
        label="🌱 Tell Carbon Coach what you did today",
        placeholder="Example: I travelled 12 km by bus today."
    ),
    outputs=gr.Textbox(
        label="🤖 Carbon Coach"
    ),
    title="🌱 Carbon Coach",
    description="Measure your carbon footprint and discover ways to reduce it.",
    examples=[
        ["I travelled 12 km by bus today."],
        ["I used 4 kWh of electricity today."],
        ["I ate chicken for lunch."],
        ["I generated 2 kg of general waste."],
        ["How can I reduce my carbon footprint?"]
    ]
)

print("✅ Carbon Coach UI connected!")

✅ Carbon Coach UI connected!


/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning: Expected 2 arguments for function <function carbon_coach_chat at 0x7b4e4f498220>, received 1.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1383: UserWarning: Expected at least 2 arguments for function <function carbon_coach_chat at 0x7b4e4f498220>, received 1.
  warnings.warn(


In [89]:

async def carbon_coach_chat(user_message):

    message = types.Content(
        role="user",
        parts=[
            types.Part(text=user_message)
        ]
    )

    response_text = ""

    try:
        async for event in runner.run_async(
            user_id="demo_user",
            session_id=session.id,
            new_message=message
        ):
            if event.is_final_response():
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            response_text += part.text

        return response_text if response_text else "⚠️ No response received."

    except Exception as e:
        return f"⚠️ Carbon Coach error: {str(e)}"


demo = gr.Interface(
    fn=carbon_coach_chat,
    inputs=gr.Textbox(
        label="🌱 Tell Carbon Coach what you did today",
        placeholder="Example: I travelled 12 km by bus today."
    ),
    outputs=gr.Textbox(
        label="🤖 Carbon Coach"
    ),
    title="🌱 Carbon Coach",
    description="Measure your carbon footprint and discover ways to reduce it.",
    examples=[
        ["I travelled 12 km by bus today."],
        ["I used 4 kWh of electricity today."],
        ["I ate chicken for lunch."],
        ["I generated 2 kg of general waste."],
        ["How can I reduce my carbon footprint?"]
    ]
)

print("✅ Gradio function fixed!")

✅ Gradio function fixed!


In [ ]:

demo.launch(
    share=True,
    debug=True,
    show_error=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d95e2d9c85116195fe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

import asyncio
import gradio as gr

def carbon_coach_chat(user_message):

    async def run_agent():
        message = types.Content(
            role="user",
            parts=[
                types.Part(text=user_message)
            ]
        )

        response_text = ""

        async for event in runner.run_async(
            user_id="demo_user",
            session_id=session.id,
            new_message=message
        ):
            if event.is_final_response():
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            response_text += part.text

        return response_text

    try:
        return asyncio.run(run_agent())

    except Exception as e:
        return f"⚠️ Error: {type(e).__name__}: {e}"


demo = gr.Interface(
    fn=carbon_coach_chat,
    inputs=gr.Textbox(
        label="🌱 Your Activity",
        placeholder="I travelled 12 km by bus today."
    ),
    outputs=gr.Textbox(
        label="🤖 Carbon Coach"
    ),
    title="🌱 Carbon Coach",
    description="AI-powered carbon footprint coach",
    submit_btn="Calculate 🌱"
)

print("✅ New Carbon Coach interface created!")

In [ ]:

demo.launch(
    share=True,
    debug=True,
    show_error=True
)

In [ ]:
result = await carbon_coach_chat(
    "I travelled 12 km by bus today."
)

print(result)

In [ ]:
%%writefile carbon_coach.py

print("Carbon Coach project file")
print("Google ADK + Gemini + deterministic carbon calculation tools")

In [ ]:
!cp carbon_coach.py carbon-coach/